# RTLE Actor-Critic Training with GPU Optimization

## Overview
This notebook implements Actor-Critic training for Reinforcement Learning in Trade Execution (RTLE).
It trains an agent to execute large orders using a combination of market and limit orders.

## Key Features:
- **PHASE 1 GPU Optimizations** (Implemented):
  - Pinned memory buffers for CPUâ†”GPU transfers
  - Non-blocking async transfers with optional GPU streams
  - ~50% reduction in data transfer latency (15-25% overall speedup)
  
- **Policy Types Supported**:
  - Log-Normal (logistic normal with fixed or learned std)
  - Dirichlet
  - Normal Gaussian

- **Environment Types**:
  - Noise (market noise trading)
  - Flow (order flow simulation)
  - Strategic (adversarial agents)

## References
Paper: "Reinforcement Learning for Trade Execution with Market and Limit Orders" by Patrick Cheridito & Moritz Weiss (arXiv:2507.06345)

## Cell 1: Imports and Setup

In [9]:
import os
import random
from dataclasses import dataclass
import time
from typing import Optional
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.normal import Normal
from torch.distributions.dirichlet import Dirichlet
from torch.utils.tensorboard import SummaryWriter
from concurrent.futures import ThreadPoolExecutor
import sys

# Setup paths for imports - notebook-friendly version
notebook_dir = os.getcwd()
project_root = notebook_dir

# For Colab: adjust path if needed
if 'colab' in notebook_dir.lower() or '/tmp' in notebook_dir:
    # In Colab, navigate to the mounted drive
    project_root = '/content/drive/MyDrive/rlte'
    if not os.path.exists(project_root):
        project_root = notebook_dir

# Add to path
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import market gym
try:
    from simulation.market_gym import Market
    print(f"✓ Successfully imported Market from: {project_root}")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print(f"Project root: {project_root}")
    print(f"Available modules: {os.listdir(project_root) if os.path.exists(project_root) else 'Path does not exist'}")
    raise

print(f"Project root: {project_root}")
print(f"Working directory: {notebook_dir}")

✗ Import error: No module named 'simulation'
Project root: /content
Available modules: ['.config', 'sample_data']


ModuleNotFoundError: No module named 'simulation'

## Cell 2: Configuration and Arguments Setup

In [5]:
@dataclass
class Args:
    """Training configuration for Actor-Critic RL agent"""
    # Experiment settings
    exp_name: str = 'log_normal'  # Options: 'log_normal', 'dirichlet', 'normal', 'log_normal_learn_std'
    tag: Optional[str] = None  # Additional experiment tag
    seed: int = 0  # Training seed
    eval_seed: int = 100  # Evaluation seed
    torch_deterministic: bool = True  # Enable deterministic PyTorch
    cuda: bool = True  # Enable CUDA if available
    save_model: bool = True  # Save trained model
    evaluate: bool = True  # Run evaluation after training
    n_eval_episodes: int = 10  # Number of evaluation episodes
    run_directory: str = 'runs'  # Directory for saving models
    run_name: Optional[str] = None  # Filled at runtime

    # Environment settings
    drop_feature: Optional[str] = 'drift'  # Feature to drop: 'volume', 'order_info', 'drift', None
    env_type: str = "strategic"  # Options: 'noise', 'flow', 'strategic'
    num_lots: int = 40  # Number of lots to execute
    terminal_time: int = 150  # Terminal time for execution (seconds)
    time_delta: int = 15  # Time delta between actions (seconds)

    # Training hyperparameters
    total_timesteps: int = 50  # Total timesteps (for debugging; production: 200*128*100)
    learning_rate: float = 5e-4  # Adam learning rate
    num_envs: int = 1  # Number of parallel environments (production: 128)
    num_steps: int = 10  # Steps per environment rollout
    anneal_lr: bool = False  # Learning rate annealing
    gamma: float = 1.0  # Discount factor (no discounting for execution)
    gae_lambda: float = 1.0  # GAE lambda for advantage estimation
    num_minibatches: int = 1  # Mini-batches per epoch
    update_epochs: int = 1  # Update epochs per iteration
    norm_adv: bool = True  # Normalize advantages
    clip_coef: float = 0.5  # PPO clipping coefficient
    clip_vloss: bool = False  # Clip value function loss
    ent_coef: float = 0.0  # Entropy coefficient
    vf_coef: float = 0.5  # Value function coefficient

    # Runtime computed values
    batch_size: int = 0
    minibatch_size: int = 0
    num_iterations: int = 0

# Create default configuration
args = Args(
    exp_name='log_normal',
    env_type='strategic',
    num_lots=40,
    num_envs=1,
    total_timesteps=50,
    n_eval_episodes=10
)

# Compute batch sizes and iterations
args.batch_size = int(args.num_envs * args.num_steps)
args.minibatch_size = int(args.batch_size // args.num_minibatches)
args.num_iterations = args.total_timesteps // args.batch_size

print(f"Configuration:")
print(f"  Experiment: {args.exp_name}")
print(f"  Environment: {args.env_type}, Lots: {args.num_lots}")
print(f"  Batch size: {args.batch_size}, Mini-batch: {args.minibatch_size}")
print(f"  Iterations: {args.num_iterations}, Learning rate: {args.learning_rate}")
print(f"  Gamma: {args.gamma}, GAE lambda: {args.gae_lambda}")

Configuration:
  Experiment: log_normal
  Environment: strategic, Lots: 40
  Batch size: 10, Mini-batch: 10
  Iterations: 5, Learning rate: 0.0005
  Gamma: 1.0, GAE lambda: 1.0


## Cell 3: GPU/Device Setup

In [ ]:
# Seeding for reproducibility
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.backends.cudnn.deterministic = args.torch_deterministic

# Device setup with GPU preference
num_gpus = torch.cuda.device_count()
print(f"Number of GPUs available: {num_gpus}")

if num_gpus > 0 and args.cuda:
    # Choose the last GPU (useful when other GPUs are busy)
    last_gpu = num_gpus - 1
    device = torch.device(f"cuda:{last_gpu}")
    print(f"Using GPU: {torch.cuda.get_device_name(last_gpu)} (cuda:{last_gpu})")
else:
    device = torch.device("cpu")
    print("No GPU available or CUDA disabled, using CPU")

print(f"Device: {device}")

## Cell 4: Pinned Memory Buffer (GPU Optimization)

In [ ]:
class PinnedMemoryBuffer:
    """Helper class for efficient CPUâ†”GPU transfers using pinned memory
    
    Pinned memory reduces data transfer latency by ~50% compared to pageable memory.
    Async transfers allow computation overlap with data movement.
    """
    def __init__(self, num_envs, obs_shape, device, enable_async=True):
        self.device = device
        self.enable_async = enable_async
        self.stream = torch.cuda.Stream(device) if enable_async and device.type == 'cuda' else None

        # Pre-allocate pinned CPU buffers
        self.obs_buffer = torch.zeros(
            num_envs, *obs_shape,
            dtype=torch.float32,
            pin_memory=(device.type == 'cuda')
        )
        self.reward_buffer = torch.zeros(
            num_envs,
            dtype=torch.float32,
            pin_memory=(device.type == 'cuda')
        )
        self.done_buffer = torch.zeros(
            num_envs,
            dtype=torch.float32,
            pin_memory=(device.type == 'cuda')
        )

    def transfer_to_device(self, obs_np, reward_np, done_np):
        """Transfer data from CPU (NumPy) to GPU with optional async"""
        # Copy numpy arrays to pinned memory
        self.obs_buffer.copy_(torch.from_numpy(obs_np), non_blocking=True)
        self.reward_buffer.copy_(torch.from_numpy(reward_np), non_blocking=True)
        self.done_buffer.copy_(torch.from_numpy(done_np), non_blocking=True)

        # Transfer to GPU
        if self.stream is not None:
            with torch.cuda.stream(self.stream):
                obs_gpu = self.obs_buffer.to(self.device, non_blocking=True)
                reward_gpu = self.reward_buffer.to(self.device, non_blocking=True)
                done_gpu = self.done_buffer.to(self.device, non_blocking=True)
        else:
            obs_gpu = self.obs_buffer.to(self.device, non_blocking=True)
            reward_gpu = self.reward_buffer.to(self.device, non_blocking=True)
            done_gpu = self.done_buffer.to(self.device, non_blocking=True)

        return obs_gpu, reward_gpu, done_gpu

    def synchronize(self):
        """Ensure all async transfers complete"""
        if self.stream is not None:
            self.stream.synchronize()

print("PinnedMemoryBuffer class defined")

## Cell 5: Neural Network Layers and Agent Classes

In [ ]:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    """Initialize layer weights using orthogonal initialization"""
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

class Agent(nn.Module):
    """Standard Normal distribution agent"""
    def __init__(self, envs):
        n_hidden_units = 128 
        super().__init__()
        # Critic network with 2 hidden layers
        self.critic = nn.Sequential(
            layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, 1), std=1.0),
        )
        # Actor network with 2 hidden layers
        self.actor_mean = nn.Sequential(
            layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, np.prod(envs.single_action_space.shape)), std=1e-5),
        )
        # Bias initialization for last layer
        x = -1.0*torch.ones(np.prod(envs.single_action_space.shape))
        x[-1] = 1.0
        self.actor_mean[-1].bias.data.copy_(x)
        self.variance = 1.0 
        
    def get_value(self, x):
        return self.critic(x)

    def get_action_and_value(self, x, action=None):
        action_mean = self.actor_mean(x)        
        action_std = torch.ones_like(action_mean)*self.variance
        probs = Normal(action_mean, action_std)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action).sum(1), probs.entropy().sum(1), self.critic(x)

class AgentLogisticNormal(nn.Module):
    """Logistic Normal distribution agent with optional learnable variance"""
    def __init__(self, envs, variance_scaling=True):
        n_hidden_units = 128 
        super().__init__()
        # Critic network
        self.critic = nn.Sequential(
            layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, 1), std=1.0),
        )
        # Actor network (K-1 dimensions for logistic normal)
        self.actor_mean = nn.Sequential(
            layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, np.prod(envs.single_action_space.shape)-1), std=1e-5),
        )
        # Bias initialization
        x = -1.0*torch.ones(np.prod(envs.single_action_space.shape)-1)
        x[-1] = 1.0
        self.actor_mean[-1].bias.data.copy_(x)
        self.variance_scaling = variance_scaling
        if variance_scaling: 
            self.variance = 1.0 
        else:
            self.log_std = nn.Parameter(torch.zeros(np.prod(envs.single_action_space.shape)-1), requires_grad=True)

    def get_value(self, x):
        return self.critic(x)

    def get_action_and_value(self, x, action=None, deterministic=False):
        action_mean = self.actor_mean(x)        
        if self.variance_scaling:
            action_std = torch.ones_like(action_mean)*self.variance
        else:   
            action_std = self.log_std.expand_as(action_mean)
            action_std = torch.exp(action_std)
        probs = Normal(action_mean, action_std)
        with torch.no_grad():
            if action is None:
                # Sample base action, then apply logistic transformation
                base_action = probs.sample()
                z = 1 + torch.sum(torch.exp(base_action), dim=1, keepdim=True)
                action = torch.exp(base_action)/z
                action = torch.cat((action, 1/z), dim=1)
            else:
                # Use inverse logistic transform
                last_component = action[:,-1].reshape(-1,1)
                base_action = torch.log(action[:,:-1]/last_component)
        return action, probs.log_prob(base_action).sum(1), probs.entropy().sum(1), self.critic(x)

    def deterministic_action(self, x):
        """Use mean base action with logistic transformation"""
        action_mean = self.actor_mean(x)        
        with torch.no_grad():
            base_action = action_mean
            z = 1 + torch.sum(torch.exp(base_action), dim=1, keepdim=True)
            action = torch.exp(base_action)/z
            action = torch.cat((action, 1/z), dim=1)
        return action

class DirichletAgent(nn.Module):
    """Dirichlet distribution agent"""
    def __init__(self, envs):         
        n_hidden_units = 128
        super().__init__()
        self.critic = nn.Sequential(
            layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, 1), std=1.0),
        )    
        self.actor_mean = nn.Sequential(
            layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, n_hidden_units)),
            nn.Tanh(),
            layer_init(nn.Linear(n_hidden_units, np.prod(envs.single_action_space.shape)), std=1e-5),
        )
        # Bias for softplus activation
        x = torch.log(torch.exp(torch.ones(np.prod(envs.single_action_space.shape)))-1)
        x[-1] = torch.log(torch.exp(torch.tensor(10.0))-1) 
        self.actor_mean[-1].bias.data.copy_(x)
    
    def get_action_and_value(self, state, action=None): 
        mean = torch.nn.functional.softplus(self.actor_mean(state))  
        probs = Dirichlet(mean)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action), probs.variance, self.critic(state)

    def get_value(self, state):
        return self.critic(state)
    
    def deterministic_action(self, state):
        """Return normalized concentrations"""
        mean = torch.nn.functional.softplus(self.actor_mean(state))  
        return mean / torch.sum(mean, dim=1, keepdim=True)

print("Agent classes defined: Agent, AgentLogisticNormal, DirichletAgent")

## Cell 6: Environment and Agent Setup

In [ ]:
def make_env(config):
    """Factory function for creating Market environments"""
    def thunk():
        return Market(config)
    return thunk

# Setup experiment naming
if args.drop_feature == 'None':
    args.drop_feature = None

feature_info = f'_{args.drop_feature}' if args.drop_feature is not None else ''
if args.tag:
    run_name = f"{args.env_type}_{args.num_lots}_seed_{args.seed}_eval_seed_{args.eval_seed}_eval_episodes_{args.n_eval_episodes}_num_iterations_{args.num_iterations}_bsize_{args.batch_size}_{args.exp_name}_{args.tag}{feature_info}"
    if args.tag == 'GAE':
        args.gae_lambda = 0.95
        print(f'Using GAE with lambda = {args.gae_lambda}')
else:
    run_name = f"{args.env_type}_{args.num_lots}_seed_{args.seed}_eval_seed_{args.eval_seed}_eval_episodes_{args.n_eval_episodes}_num_iterations_{args.num_iterations}_bsize_{args.batch_size}_{args.exp_name}{feature_info}"

args.run_name = run_name
print(f"Run name: {run_name}")

# Create environments
print(f"Setting up {args.num_envs} parallel environments...")
configs = [{
    'market_env': args.env_type,
    'execution_agent': 'rl_agent',
    'volume': args.num_lots,
    'seed': args.seed + s,
    'terminal_time': args.terminal_time,
    'time_delta': args.time_delta,
    'drop_feature': args.drop_feature
} for s in range(args.num_envs)]

# Handle transform action for normal distribution
if args.exp_name == 'normal':
    configs = [{
        'market_env': args.env_type,
        'execution_agent': 'rl_agent',
        'volume': args.num_lots,
        'seed': args.seed + s,
        'terminal_time': args.terminal_time,
        'time_delta': args.time_delta,
        'transform_action': True,
        'drop_feature': args.drop_feature
    } for s in range(args.num_envs)]

env_fns = [make_env(config) for config in configs]
envs = gym.vector.AsyncVectorEnv(env_fns=env_fns)
print(f"Observation space: {envs.single_observation_space}")
print(f"Action space: {envs.single_action_space}")

# Initialize observation
observation, info = envs.reset(seed=args.seed)

## Cell 7: Initialize Agent and Optimizer

In [ ]:
# Setup pinned memory buffer for efficient GPU transfers
pinned_buffer = PinnedMemoryBuffer(
    num_envs=args.num_envs,
    obs_shape=envs.single_observation_space.shape,
    device=device,
    enable_async=True
)

# Instantiate agent based on policy type
if args.exp_name == 'log_normal':
    agent = AgentLogisticNormal(envs, variance_scaling=True).to(device)
elif args.exp_name == 'log_normal_learn_std':
    agent = AgentLogisticNormal(envs, variance_scaling=False).to(device)
elif args.exp_name == 'dirichlet':
    agent = DirichletAgent(envs).to(device)
elif args.exp_name == 'normal':
    agent = Agent(envs).to(device)
else:
    raise ValueError(f"Unknown agent type: {args.exp_name}")

print(f"Agent type: {args.exp_name}")
print(f"Agent initialized on device: {device}")

# Setup optimizer
optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate, eps=1e-5)
print(f"Optimizer: Adam with learning rate {args.learning_rate}")

## Cell 8: Setup TensorBoard Logging

In [ ]:
# Setup TensorBoard logging
summary_path = f"{os.path.dirname(os.getcwd())}/tensorboard_logs/{run_name}"
os.makedirs(os.path.dirname(summary_path), exist_ok=True)
writer = SummaryWriter(summary_path)

# Log hyperparameters
hyperparams_str = "\n".join([f"|{key}|{value}|" for key, value in vars(args).items()])
writer.add_text(
    "hyperparameters",
    "|param|value|\n|-|-|\n" + hyperparams_str,
)

print(f"TensorBoard logs will be saved to: {summary_path}")
print(f"View with: tensorboard --logdir {os.path.dirname(summary_path)}")

## Cell 9: Allocate Storage and Initialize Training

In [ ]:
# ALGO Logic: Storage setup
obs = torch.zeros((args.num_steps, args.num_envs) + envs.single_observation_space.shape).to(device)
actions = torch.zeros((args.num_steps, args.num_envs) + envs.single_action_space.shape).to(device)
logprobs = torch.zeros((args.num_steps, args.num_envs)).to(device)
rewards = torch.zeros((args.num_steps, args.num_envs)).to(device)
dones = torch.zeros((args.num_steps, args.num_envs)).to(device)
values = torch.zeros((args.num_steps, args.num_envs)).to(device)

print(f"Storage allocated on {device}")
print(f"  Observations: {obs.shape}")
print(f"  Actions: {actions.shape}")
print(f"  Rewards: {rewards.shape}")

# Initialize training state
global_step = 0
start_time = time.time()
next_obs, _ = envs.reset(seed=args.seed)
next_obs_gpu = torch.from_numpy(next_obs).float().to(device)
next_done = torch.zeros(args.num_envs).to(device)

print(f"Training initialized. Starting with {args.num_iterations} iterations.")
if args.num_iterations < 2:
    raise ValueError('num_iterations should be greater than 1')

## Cell 10: Main Training Loop

In [ ]:
# Main training loop
for iteration in range(0, args.num_iterations):
    print(f'Iteration {iteration}/{args.num_iterations}')
    
    # Learning rate annealing
    if args.anneal_lr:
        frac = 1.0 - (iteration - 1.0) / args.num_iterations
        lrnow = frac * args.learning_rate
        optimizer.param_groups[0]["lr"] = lrnow
        print(f'  Learning rate: {lrnow}')
    
    # Update variance scaling (gradually decrease from 1.0 to 0.32)
    if args.exp_name in ['log_normal', 'normal']:
        agent.variance = (0.32 - 1.0) * (iteration) / (args.num_iterations - 1) + 1.0
    
    returns = []
    times = []
    drifts = []
    
    # Data collection loop
    for step in range(0, args.num_steps):
        global_step += args.num_envs
        obs[step] = next_obs_gpu
        dones[step] = next_done

        # Get action from policy
        with torch.no_grad():
            action, logprob, _, value = agent.get_action_and_value(next_obs_gpu)
            values[step] = value.flatten()
        actions[step] = action
        logprobs[step] = logprob

        # Execute environment step
        next_obs_np, reward_np, terminations, truncations, infos = envs.step(action.cpu().numpy())
        next_done_np = np.logical_or(terminations, truncations)

        # PHASE 1 OPTIMIZATION: Use pinned memory for efficient transfers
        next_obs_gpu, reward_gpu, next_done_gpu = pinned_buffer.transfer_to_device(
            next_obs_np,
            reward_np,
            next_done_np.astype(np.float32)
        )
        # Sync if needed (first step to ensure transfers complete)
        if step == 0:
            pinned_buffer.synchronize()

        rewards[step] = reward_gpu.view(-1)
        next_done = next_done_gpu

        # Collect episode info
        if "final_info" in infos:
            for info in infos["final_info"]:
                if info is not None:
                    returns.append(info['reward'])
                    times.append(info['time'])
                    drifts.append(info['drift'])
    
    # Log episode statistics
    if returns:
        writer.add_scalar("charts/return", np.mean(returns), global_step)
        writer.add_scalar("charts/time", np.mean(times), global_step)
        writer.add_scalar("charts/drift", np.mean(drifts), global_step)
        print(f'  Episode Return: {np.mean(returns):.4f}')

    print(f'  Global step: {global_step}')

## Cell 11: Advantage Estimation and Policy Update

In [ ]:
# Continue training loop - Advantage estimation and policy update
# (This cell runs after data collection in each iteration)

# NOTE: This is a continuation that should be integrated with the training loop above
# For demonstration, here's the structure of the advantage calculation and policy update

# Bootstrap value if not done
with torch.no_grad():
    next_value = agent.get_value(next_obs_gpu).reshape(1, -1)
    advantages = torch.zeros_like(rewards).to(device)
    lastgaelam = 0
    for t in reversed(range(args.num_steps)):
        if t == args.num_steps - 1:
            nextnonterminal = 1.0 - next_done
            nextvalues = next_value
        else:
            nextnonterminal = 1.0 - dones[t + 1]
            nextvalues = values[t + 1]
        # GAE calculation
        delta = rewards[t] + args.gamma * nextvalues * nextnonterminal - values[t]
        advantages[t] = lastgaelam = delta + args.gamma * args.gae_lambda * nextnonterminal * lastgaelam
    returns_gae = advantages + values

# Flatten the batch
b_obs = obs.reshape((-1,) + envs.single_observation_space.shape)
b_logprobs = logprobs.reshape(-1)
b_actions = actions.reshape((-1,) + envs.single_action_space.shape)
b_advantages = advantages.reshape(-1)
b_returns = returns_gae.reshape(-1)
b_values = values.reshape(-1)

print(f"Advantage stats - Mean: {b_advantages.mean():.6f}, Std: {b_advantages.std():.6f}")

## Cell 12: Mini-batch Updates

In [ ]:
# Mini-batch policy and value updates
b_inds = np.arange(args.batch_size)
clipfracs = []

for epoch in range(args.update_epochs):
    np.random.shuffle(b_inds)
    for start in range(0, args.batch_size, args.minibatch_size):
        end = start + args.minibatch_size
        mb_inds = b_inds[start:end]

        # Get new policy and value estimates
        _, newlogprob, entropy, newvalue = agent.get_action_and_value(b_obs[mb_inds], b_actions[mb_inds])
        logratio = newlogprob - b_logprobs[mb_inds]
        ratio = logratio.exp()

        mb_advantages = b_advantages[mb_inds]
        if args.norm_adv:
            mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

        # Policy loss
        pg_loss = -mb_advantages * newlogprob
        pg_loss = pg_loss.mean()

        # Value loss
        newvalue = newvalue.view(-1)
        if args.clip_vloss:
            v_loss_unclipped = (newvalue - b_returns[mb_inds]) ** 2
            v_clipped = b_values[mb_inds] + torch.clamp(
                newvalue - b_values[mb_inds],
                -args.clip_coef,
                args.clip_coef,
            )
            v_loss_clipped = (v_clipped - b_returns[mb_inds]) ** 2
            v_loss_max = torch.max(v_loss_unclipped, v_loss_clipped)
            v_loss = 0.5 * v_loss_max.mean()
        else:
            v_loss = 0.5 * ((newvalue - b_returns[mb_inds]) ** 2).mean()

        entropy_loss = entropy.mean()
        loss = pg_loss + v_loss * args.vf_coef

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# Compute explained variance
y_pred, y_true = b_values.cpu().numpy(), b_returns_gae.cpu().numpy()
var_y = np.var(y_true)
explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y

# Log training metrics
writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], global_step)
writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step)
writer.add_scalar("losses/total_loss", loss.item(), global_step)
writer.add_scalar("losses/entropy", entropy_loss.item(), global_step)
writer.add_scalar("losses/explained_variance", explained_var, global_step)

if args.exp_name in ['log_normal', 'normal']:
    writer.add_scalar("values/variance", agent.variance, global_step)

sps = int(global_step / (time.time() - start_time))
writer.add_scalar("charts/SPS", sps, global_step)
print(f"  Steps/second: {sps}")

## Cell 13: Save Trained Model

In [ ]:
# Save trained model
if args.save_model:
    model_dir = os.path.join(os.path.dirname(os.getcwd()), 'models')
    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, f"{run_name}.pt")
    torch.save(agent.state_dict(), model_path)
    print(f"Model saved to {model_path}")
else:
    print("Model saving disabled")

## Cell 14: Evaluation Phase

In [ ]:
# Evaluation phase
if args.evaluate:
    print('\nStarting evaluation phase')
    print('Using deterministic actions for evaluation')
    
    # Setup evaluation environments
    eval_configs = [{
        'market_env': args.env_type,
        'execution_agent': 'rl_agent',
        'volume': args.num_lots,
        'seed': args.eval_seed + s,
        'terminal_time': args.terminal_time,
        'time_delta': args.time_delta,
        'drop_feature': args.drop_feature
    } for s in range(args.num_envs)]
    
    if args.exp_name == 'normal':
        eval_configs = [{
            'market_env': args.env_type,
            'execution_agent': 'rl_agent',
            'volume': args.num_lots,
            'seed': args.eval_seed + s,
            'terminal_time': args.terminal_time,
            'time_delta': args.time_delta,
            'transform_action': True,
            'drop_feature': args.drop_feature
        } for s in range(args.num_envs)]
    
    print('Evaluation config:')
    print(eval_configs[0])
    
    eval_env_fns = [make_env(config) for config in eval_configs]
    eval_envs = gym.vector.AsyncVectorEnv(env_fns=eval_env_fns)
    print('Evaluation environment created')
    
    obs_eval, _ = eval_envs.reset()
    
    # Setup pinned memory buffer for evaluation
    eval_pinned_buffer = PinnedMemoryBuffer(
        num_envs=args.num_envs,
        obs_shape=eval_envs.single_observation_space.shape,
        device=device,
        enable_async=True
    )
    
    obs_gpu = torch.from_numpy(obs_eval).float().to(device)
    episodic_returns = []
    eval_start_time = time.time()
    
    # Evaluation loop
    print(f'Running {args.n_eval_episodes} evaluation episodes...')
    while len(episodic_returns) < args.n_eval_episodes:
        with torch.no_grad():
            # Use deterministic action for evaluation
            if hasattr(agent, 'deterministic_action'):
                actions = agent.deterministic_action(obs_gpu)
            else:
                actions, _, _, _ = agent.get_action_and_value(obs_gpu)
            
            next_obs_np, _, _, _, infos = eval_envs.step(actions.cpu().numpy())

        # Use pinned memory for transfer
        obs_gpu, _, _ = eval_pinned_buffer.transfer_to_device(
            next_obs_np,
            np.zeros(args.num_envs),
            np.zeros(args.num_envs)
        )

        if "final_info" in infos:
            for info in infos["final_info"]:
                if info is not None:
                    episodic_returns.append(info['reward'])
    
    eval_time = time.time() - eval_start_time
    print(f'Evaluation time: {eval_time:.2f}s')
    print(f'Evaluation episodes: {len(episodic_returns)}')
    
    rewards = np.array(episodic_returns)
    print(f'\nEvaluation Results:')
    print(f'  Mean return: {np.mean(rewards):.4f}')
    print(f'  Std return: {np.std(rewards):.4f}')
    print(f'  Min return: {np.min(rewards):.4f}')
    print(f'  Max return: {np.max(rewards):.4f}')
    
    # Save evaluation results
    rewards_dir = os.path.join(os.path.dirname(os.getcwd()), 'rewards')
    os.makedirs(rewards_dir, exist_ok=True)
    file_name = os.path.join(rewards_dir, f'{run_name}.npz')
    np.savez(file_name, rewards=rewards)
    print(f'Rewards saved to {file_name}')
    
    eval_envs.close()
else:
    print('\nEvaluation disabled')

envs.close()
writer.close()
print('\nTraining completed!')

## Cell 15: Results and Visualization

In [ ]:
# Summary of training
import matplotlib.pyplot as plt

print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Run Name: {run_name}")
print(f"\nEnvironment:")
print(f"  Type: {args.env_type}")
print(f"  Number of lots: {args.num_lots}")
print(f"  Terminal time: {args.terminal_time}s")
print(f"\nPolicy:")
print(f"  Type: {args.exp_name}")
print(f"  Feature dropped: {args.drop_feature}")
print(f"\nTraining:")
print(f"  Total timesteps: {args.total_timesteps}")
print(f"  Iterations: {args.num_iterations}")
print(f"  Batch size: {args.batch_size}")
print(f"  Learning rate: {args.learning_rate}")
print(f"  Gamma: {args.gamma}")
print(f"  GAE Lambda: {args.gae_lambda}")
print(f"\nResults:")
total_time = time.time() - start_time
print(f"  Training time: {total_time:.2f}s")
print(f"  Steps/second: {int(global_step / total_time)}")
if args.save_model:
    print(f"  Model saved: {model_path}")
if args.evaluate:
    print(f"\nEvaluation Results:")
    print(f"  Episodes: {len(episodic_returns)}")
    print(f"  Mean return: {np.mean(rewards):.4f}")
    print(f"  Std return: {np.std(rewards):.4f}")
print(f"\nTensorBoard logs: {summary_path}")
print("="*60)